## Install Python 3.8

In [ ]:
!apt-get update -qq
!apt-get install -y -qq make build-essential libssl-dev zlib1g-dev \
libbz2-dev libreadline-dev libsqlite3-dev wget curl llvm \
libncursesw5-dev xz-utils tk-dev libxml2-dev libxmlsec1-dev \
libffi-dev liblzma-dev ffmpeg

!curl https://pyenv.run | bash

import os
os.environ["PYENV_ROOT"] = "/root/.pyenv"
os.environ["PATH"] = "/root/.pyenv/bin:/root/.pyenv/shims:" + os.environ["PATH"]

!pyenv install 3.8.18
!pyenv global 3.8.18

!/root/.pyenv/versions/3.8.18/bin/python -m venv /content/py38

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   270  100   270    0     0   1555      0 --:--:-- --:--:-- --:--:--  1560


pyenv: /root/.pyenv/versions/3.8.18 already exists
continue with installation? (y/N) N


In [ ]:
import os
os.environ["PYENV_ROOT"] = "/root/.pyenv"
os.environ["PATH"] = "/root/.pyenv/bin:/root/.pyenv/shims:" + os.environ["PATH"]

In [ ]:
!/root/.pyenv/versions/3.8.18/bin/python --version

!/root/.pyenv/versions/3.8.18/bin/python -m venv /content/py38

!/content/py38/bin/python --version

Python 3.8.18
Python 3.8.18


## Download Libraries

In [ ]:
!/content/py38/bin/pip install \
numpy==1.23.5 \
omegaconf==2.0.6 hydra-core==1.0.7 scipy==1.9.3 \
sentencepiece==0.1.99 python_speech_features==0.6 \
scikit-video==1.1.11 opencv-python-headless==4.7.0.72 \
mediapipe==0.10.9 \
openai-whisper fastapi uvicorn pyngrok nest_asyncio \
python-multipart werpy contractions textsearch \
scikit-learn joblib librosa rtdl_revisiting_models

In [ ]:
!/content/py38/bin/pip install \
torch==1.13.1+cu116 torchvision==0.14.1+cu116 torchaudio==0.13.1 \
--extra-index-url https://download.pytorch.org/whl/cu116

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu116


In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/av_hubert.git

%cd /content/av_hubert
!git submodule update --init --recursive

/content
fatal: destination path 'av_hubert' already exists and is not an empty directory.
/content/av_hubert


In [ ]:
%cd /content/av_hubert/fairseq
!/content/py38/bin/pip install --editable .

/content/av_hubert/fairseq
Obtaining file:///content/av_hubert/fairseq
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for fairseq (pyproject.toml) ... done
  Created wheel for fairseq: filename=fairseq-1.0.0a0+afc77bd-0.editable-cp38-cp38-linux_x86_64.whl size=8850 sha256=ca0aeebe2cd67266b3a1632bc16ff49966ae1131485e59f0e72e1b119ff9c963
  Stored in directory: /tmp/pip-ephem-wheel-cache-nj6dxvm7/wheels/c0/bc/12/b44810d3f46a1b839bfe22323dd4609d9b0c8430ca1e0cc32e
Successfully built fairseq
  Attempting uninstall: fairseq
    Found existing installation: fairseq 1.0.0a0+afc77bd
    Uninstalling fairseq-1.0.0a0+afc77bd:
      Successfully uninstalled fairseq-1.0.0a0+afc77bd


In [ ]:
!mkdir -p /content/data

!wget https://dl.fbaipublicfiles.com/avhubert/model/lrs3_vox/vsr/base_vox_433h.pt \
-O /content/data/finetune-model.pt

--2026-05-09 18:32:30--  https://dl.fbaipublicfiles.com/avhubert/model/lrs3_vox/vsr/base_vox_433h.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.171.22.13, 3.171.22.118, 3.171.22.33, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.171.22.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1928063847 (1.8G) [binary/octet-stream]
Saving to: ‘/content/data/finetune-model.pt’

/content/data/finet 100%[===================>]   1.79G   142MB/s    in 14s     

2026-05-09 18:32:44 (131 MB/s) - ‘/content/data/finetune-model.pt’ saved [1928063847/1928063847]



## Prediction file


In [ ]:
%%writefile /content/run.py
import sys

# Hack for duplicate model
if len(sys.argv) == 1:
    sys.argv.append("dummy_arg_to_prevent_error")

import cv2
import tempfile
import torch
from argparse import Namespace
from fairseq import checkpoint_utils, tasks, utils
from fairseq.dataclass.configs import GenerationConfig

sys.path.insert(0, "/content/av_hubert/avhubert")
sys.modules['fairseq.metrics'] = __import__('fairseq.logging.metrics')

def predict(video_path, ckpt_path, user_dir):
    num_frames = int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_COUNT))
    data_dir = tempfile.mkdtemp()

    with open(f"{data_dir}/test.tsv", "w") as f:
        f.write("/\n" + f"test-0\t{video_path}\t{None}\t{num_frames}\t{int(16000*num_frames/25)}\n")
    with open(f"{data_dir}/test.wrd", "w") as f:
        f.write("DUMMY\n")

    utils.import_user_module(Namespace(user_dir=user_dir))
    gen_cfg = GenerationConfig(beam=20)
    models, saved_cfg, task = checkpoint_utils.load_model_ensemble_and_task([ckpt_path])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    models = [model.eval().to(device) for model in models]

    saved_cfg.task.modalities = ["video"]
    saved_cfg.task.data = data_dir
    saved_cfg.task.label_dir = data_dir

    task = tasks.setup_task(saved_cfg.task)
    task.load_dataset("test", task_cfg=saved_cfg.task)
    generator = task.build_generator(models, gen_cfg)

    def decode_fn(x):
        dictionary = task.target_dictionary
        symbols_ignore = generator.symbols_to_strip_from_output
        symbols_ignore.add(dictionary.pad())
        return task.datasets["test"].label_processors[0].decode(x, symbols_ignore)

    itr = task.get_batch_iterator(dataset=task.dataset("test")).next_epoch_itr(shuffle=False)
    sample = next(itr)
    if torch.cuda.is_available():
        sample = utils.move_to_cuda(sample)

    hypos = task.inference_step(generator, models, sample)
    hypo = hypos[0][0]['tokens'].int().cpu()
    print("Prediction:", decode_fn(hypo))

predict("/content/data/roi.mp4", "/content/data/finetune-model.pt", "/content/av_hubert/avhubert")

Overwriting /content/run.py


## Server Creation and Running

In [ ]:
def create_server_file():
    with open("server_app.py", "w") as f:
        f.write("""
import cv2, os, re, string, subprocess, shlex, whisper, torch, contractions
import mediapipe as mp
import numpy as np
from fastapi import FastAPI, UploadFile, File, HTTPException
from werpy import wer
from difflib import SequenceMatcher
import uvicorn
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib
import pickle
from rtdl_revisiting_models import FTTransformer
from difflib import SequenceMatcher

app = FastAPI()
# load whisper moder
whisper_model = whisper.load_model("small")
# array to store mismatch segments timestamps between AV-HuBERT prediction and Whisper transcript
mismatch_segments = []

# function to process text
def preprocess_text(text):
    text = contractions.fix(text)
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

# function to calculate the number of correct words between AV-HuBERT prediction
# and Whisper Transcript
def count_correct_words(ref, hyp):
    ref_w, hyp_w = ref.split(), hyp.split()
    n, m = len(ref_w), len(hyp_w)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_w[i-1] == hyp_w[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    i, j, correct = n, m, 0
    while i > 0 and j > 0:
        if ref_w[i-1] == hyp_w[j-1]:
            correct += 1; i -= 1; j -= 1
        else:
            if dp[i-1][j] < dp[i][j-1] and dp[i-1][j] < dp[i-1][j-1]:
                i -= 1
            elif dp[i][j-1] < dp[i-1][j-1]:
                j -= 1
            else:
                i -= 1; j -= 1
    return correct

# function to preprocess video for AV-HuBERT to crop to ROI using MediaPipe
def internal_preprocess(input_path, output_path):
    # initialize MediaPipe Face Mesh
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1)

    # get the frames-per-second, height and width of the video
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # setup FFmpeg Pipe, we use libx264 and yuv420p to be compatible with AV-HuBERT
    cmd = f'ffmpeg -y -f rawvideo -vcodec rawvideo -s 96x96 -pix_fmt gray -r {fps} -i - -vcodec libx264 -pix_fmt yuv420p -crf 18 "{output_path}"'
    process = subprocess.Popen(shlex.split(cmd), stdin=subprocess.PIPE)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # detect landmarks
        res = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            cx, cy = int((lm[13].x + lm[14].x)/2 * w), int((lm[13].y + lm[14].y)/2 * h)
            y1, y2, x1, x2 = cy-48, cy+48, cx-48, cx+48
            # crop (96x96)
            crop = np.zeros((96, 96, 3), dtype=np.uint8)
            sy1, sy2 = max(0, y1), min(h, y2)
            sx1, sx2 = max(0, x1), min(w, x2)
            dy1, dy2 = max(0, -y1), 96-max(0, y2-h)
            dx1, dx2 = max(0, -x1), 96-max(0, x2-w)

            crop[dy1:dy2, dx1:dx2] = frame[sy1:sy2, sx1:sx2]
            process.stdin.write(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).tobytes())
        else:
            process.stdin.write(np.zeros((96, 96), dtype=np.uint8).tobytes())

    process.stdin.close()
    process.wait()
    cap.release()
    face_mesh.close()

# function to extract MFCC and Spectral features from audio
def extract_mfcc_features(audio_path, n_mfcc=40, n_fft=2048, hop_length=512):
    try:
        # load the audio file
        audio_data, sr = librosa.load(audio_path, sr=None)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return None

    # MFCC
    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    mfcc_mean = np.mean(mfccs.T, axis=0)


    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(y=audio_data, sr=sr)
    centroid_mean = np.mean(centroid)

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=audio_data, sr=sr)
    rolloff_mean = np.mean(rolloff)

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=audio_data, sr=sr)
    bandwidth_mean = np.mean(bandwidth)

    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(y=audio_data, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)

    features = np.hstack([mfcc_mean, centroid_mean, rolloff_mean, bandwidth_mean, contrast_mean])

    return features

# function to run the FTTransformer and get prediction
def run_classifier(similarity, wer, mfcc):
  # load the scaler, label encoder, and the best threshold
  scaler = joblib.load('/content/scaler_v1.pkl')
  le = joblib.load('/content/label_encoder_v1.pkl')
  threshold = joblib.load('/content/best_threshold_compat.pkl')
  # create the features array
  X_new = np.hstack([mfcc, wer, similarity])

  # re-initialize the model architecture
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = FTTransformer(
          n_cont_features=52,             # MFCC(50) + WER(1) + Sim(1) = 52
          cat_cardinalities=[],
          d_out=2,                        # 2 classes (0/1)
          n_blocks=3,
          d_block=128,
          attention_n_heads=8,
          attention_dropout=0.2,
          ffn_d_hidden_multiplier=4/3,
          ffn_dropout=0.1,
          residual_dropout=0.0
      ).to(device)
  # load model state
  model.load_state_dict(torch.load('/content/best_model.pt', map_location=device))
  model.eval()

  # scale the data
  X_new_scaled = scaler.transform(X_new.reshape(1, -1))

  # get results
  with torch.no_grad():
      logits = model(torch.tensor(X_new_scaled).float().to(device), None)
      prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

  # apply the optimal threshold
  if (prob >= threshold):
    final_prediction = "Fake"
    confidence = prob
  else:
    final_prediction = "Real"
    confidence = 1 - prob

  return final_prediction, float(confidence)

# function to get word timestamps
def get_word_timestamps(segments):
    # array to store each word in the sentence with its start-end timestamps
    words = []

    for seg in segments:
        for w in seg["words"]:
            words.append({
                "word": w["word"].strip().lower(),
                "start": float(w["start"]),
                "end": float(w["end"])
            })

    return words


# function to detect mismatches between visual and audio text
def localize_mismatches(audio_file, visual_text, segments):
    # speech-to-text words + timestamps
    audio_words = get_word_timestamps(segments)
    # get audio and visual tokens
    audio_tokens = [w["word"] for w in audio_words]
    visual_tokens = visual_text.lower().split()
    # create a SequenceMatcher object
    matcher = SequenceMatcher(None, audio_tokens, visual_tokens)
    # array to store mismatches
    mismatches = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        # equal = matching words
        if tag == "equal":
            continue

        # Replace words
        elif tag == "replace":
            for k in range(i1, i2):
                visual_word = visual_tokens[j1 + (k - i1)] if (j1 + (k - i1)) < j2 else ""

                mismatches.append({
                    "type": "replace",
                    "audio_word": audio_words[k]["word"],
                    "visual_word": visual_word,
                    "start": audio_words[k]["start"],
                    "end": audio_words[k]["end"]
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

        # Word exists in audio but missing in visual
        elif tag == "delete":
            for k in range(i1, i2):
                mismatches.append({
                    "type": "delete",
                    "audio_word": audio_words[k]["word"],
                    "visual_word": "",
                    "start": audio_words[k]["start"],
                    "end": audio_words[k]["end"]
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

        # Word inserted in visual transcript
        elif tag == "insert":
            for k in range(j1, j2):
                mismatches.append({
                    "type": "insert",
                    "audio_word": "",
                    "visual_word": visual_tokens[k],
                    "start": None,
                    "end": None
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

    return mismatches


@app.post("/process")
async def process_video(video: UploadFile = File(...)):
    v_path = f"temp_{video.filename}"
    roi_path = "/content/data/roi.mp4"

    try:
        with open(v_path, "wb") as b:
            b.write(await video.read())

        # Whisper
        audio_out = whisper_model.transcribe(v_path, language="en", word_timestamps=True)
        whisper_text = preprocess_text(audio_out["text"])

        # AV-Hubert
        internal_preprocess(v_path, roi_path)
        av_raw = subprocess.getoutput("/content/py38/bin/python /content/run.py")

        lines = av_raw.splitlines()
        if len(lines) == 0:
            raise Exception("AV-Hubert returned empty output")

        last_line = lines[-1]

        if "Prediction:" not in last_line:
            raise Exception(f"Unexpected AV-Hubert output: {last_line}")

        av_text = preprocess_text(last_line.replace("Prediction: ", "").strip())

        # Metrics
        res_wer = wer(whisper_text, av_text) * 100
        res_sim = SequenceMatcher(None, whisper_text, av_text).ratio() * 100
        res_corr = count_correct_words(whisper_text, av_text)

        # MFCC features
        features = extract_mfcc_features(v_path)

        # run classifier
        prediction, confidence = run_classifier(round(res_sim, 2), round(res_wer, 2), features)
        # localization
        if prediction == "Fake":
          fake_segments = localize_mismatches(v_path, av_text, audio_out["segments"])
        else:
          fake_segments = []

        return {
            "whisper_text": whisper_text,
            "avhubert_text": av_text,
            "prediction": prediction,
            "confidence_score": f"{confidence:.3f}",
            "segments": fake_segments
        }

    except Exception as e:
        print("🔥 INTERNAL ERROR:", str(e))
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)



#This prevents the API server from running during testing.
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
    """)
    print("server file is created")
create_server_file()



تم تحديث ملف السيرفر بنجاح ✅


<>:29: SyntaxWarning: invalid escape sequence '\s'
<>:29: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_31111/1502034754.py:29: SyntaxWarning: invalid escape sequence '\s'
  return re.sub(r'\s+', ' ', text).strip()


In [ ]:
!pip install opencv-python whisper torch contractions mediapipe numpy fastapi uvicorn werpy librosa scikit-learn joblib pickle-mixin rtdl-revisiting-models

In [ ]:
! pip uninstall whisper -y
! pip install openai-whisper

In [ ]:
import cv2, os, re, string, subprocess, shlex, whisper, torch, contractions
import mediapipe as mp
import numpy as np
from fastapi import FastAPI, UploadFile, File, HTTPException
from werpy import wer
from difflib import SequenceMatcher
import uvicorn
import librosa
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import joblib
import pickle
from rtdl_revisiting_models import FTTransformer
from difflib import SequenceMatcher

app = FastAPI()
# load whisper moder
whisper_model = whisper.load_model("small")
# array to store mismatch segments timestamps between AV-HuBERT prediction and Whisper transcript
mismatch_segments = []

# function to process text
def preprocess_text(text):
    text = contractions.fix(text)
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return re.sub(r'\s+', ' ', text).strip()

# function to calculate the number of correct words between AV-HuBERT prediction
# and Whisper Transcript
def count_correct_words(ref, hyp):
    ref_w, hyp_w = ref.split(), hyp.split()
    n, m = len(ref_w), len(hyp_w)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            if ref_w[i-1] == hyp_w[j-1]: dp[i][j] = dp[i-1][j-1]
            else: dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    i, j, correct = n, m, 0
    while i > 0 and j > 0:
        if ref_w[i-1] == hyp_w[j-1]:
            correct += 1; i -= 1; j -= 1
        else:
            if dp[i-1][j] < dp[i][j-1] and dp[i-1][j] < dp[i-1][j-1]:
                i -= 1
            elif dp[i][j-1] < dp[i-1][j-1]:
                j -= 1
            else:
                i -= 1; j -= 1
    return correct

# function to preprocess video for AV-HuBERT to crop to ROI using MediaPipe
def internal_preprocess(input_path, output_path):
    # initialize MediaPipe Face Mesh
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1)

    # get the frames-per-second, height and width of the video
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # setup FFmpeg Pipe, we use libx264 and yuv420p to be compatible with AV-HuBERT
    cmd = f'ffmpeg -y -f rawvideo -vcodec rawvideo -s 96x96 -pix_fmt gray -r {fps} -i - -vcodec libx264 -pix_fmt yuv420p -crf 18 "{output_path}"'
    process = subprocess.Popen(shlex.split(cmd), stdin=subprocess.PIPE)

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        # detect landmarks
        res = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if res.multi_face_landmarks:
            lm = res.multi_face_landmarks[0].landmark
            cx, cy = int((lm[13].x + lm[14].x)/2 * w), int((lm[13].y + lm[14].y)/2 * h)
            y1, y2, x1, x2 = cy-48, cy+48, cx-48, cx+48
            # crop (96x96)
            crop = np.zeros((96, 96, 3), dtype=np.uint8)
            sy1, sy2 = max(0, y1), min(h, y2)
            sx1, sx2 = max(0, x1), min(w, x2)
            dy1, dy2 = max(0, -y1), 96-max(0, y2-h)
            dx1, dx2 = max(0, -x1), 96-max(0, x2-w)

            crop[dy1:dy2, dx1:dx2] = frame[sy1:sy2, sx1:sx2]
            process.stdin.write(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY).tobytes())
        else:
            process.stdin.write(np.zeros((96, 96), dtype=np.uint8).tobytes())

    process.stdin.close()
    process.wait()
    cap.release()
    face_mesh.close()

# function to extract MFCC and Spectral features from audio
def extract_mfcc_features(audio_path, n_mfcc=40, n_fft=2048, hop_length=512):
    try:
        # load the audio file
        audio_data, sr = librosa.load(audio_path, sr=None)
    except Exception as e:
        print(f"Error loading audio file {audio_path}: {e}")
        return None

    # MFCC
    mfccs = librosa.feature.mfcc(y=audio_data, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop_length)
    mfcc_mean = np.mean(mfccs.T, axis=0)


    # Spectral Centroid
    centroid = librosa.feature.spectral_centroid(y=audio_data, sr=sr)
    centroid_mean = np.mean(centroid)

    # Spectral Rolloff
    rolloff = librosa.feature.spectral_rolloff(y=audio_data, sr=sr)
    rolloff_mean = np.mean(rolloff)

    # Spectral Bandwidth
    bandwidth = librosa.feature.spectral_bandwidth(y=audio_data, sr=sr)
    bandwidth_mean = np.mean(bandwidth)

    # Spectral Contrast
    contrast = librosa.feature.spectral_contrast(y=audio_data, sr=sr)
    contrast_mean = np.mean(contrast, axis=1)

    features = np.hstack([mfcc_mean, centroid_mean, rolloff_mean, bandwidth_mean, contrast_mean])

    return features

# function to run the FTTransformer and get prediction
def run_classifier(similarity, wer, mfcc):
  # load the scaler, label encoder, and the best threshold
  scaler = joblib.load('/content/scaler_v1.pkl')
  le = joblib.load('/content/label_encoder_v1.pkl')
  threshold = joblib.load('/content/best_threshold_compat.pkl')
  # create the features array
  X_new = np.hstack([mfcc, wer, similarity])

  # re-initialize the model architecture
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model = FTTransformer(
          n_cont_features=52,             # MFCC(50) + WER(1) + Sim(1) = 52
          cat_cardinalities=[],
          d_out=2,                        # 2 classes (0/1)
          n_blocks=3,
          d_block=128,
          attention_n_heads=8,
          attention_dropout=0.2,
          ffn_d_hidden_multiplier=4/3,
          ffn_dropout=0.1,
          residual_dropout=0.0
      ).to(device)
  # load model state
  model.load_state_dict(torch.load('/content/best_model.pt', map_location=device))
  model.eval()

  # scale the data
  X_new_scaled = scaler.transform(X_new.reshape(1, -1))

  # get results
  with torch.no_grad():
      logits = model(torch.tensor(X_new_scaled).float().to(device), None)
      prob = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

  # apply the optimal threshold
  if (prob >= threshold):
    final_prediction = "Fake"
    confidence = prob
  else:
    final_prediction = "Real"
    confidence = 1 - prob

  return final_prediction, float(confidence)

# function to get word timestamps
def get_word_timestamps(segments):
    # array to store each word in the sentence with its start-end timestamps
    words = []

    for seg in segments:
        for w in seg["words"]:
            words.append({
                "word": w["word"].strip().lower(),
                "start": float(w["start"]),
                "end": float(w["end"])
            })

    return words


# function to detect mismatches between visual and audio text
def localize_mismatches(audio_file, visual_text, segments):
    # speech-to-text words + timestamps
    audio_words = get_word_timestamps(segments)
    # get audio and visual tokens
    audio_tokens = [w["word"] for w in audio_words]
    visual_tokens = visual_text.lower().split()
    # create a SequenceMatcher object
    matcher = SequenceMatcher(None, audio_tokens, visual_tokens)
    # array to store mismatches
    mismatches = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():

        # equal = matching words
        if tag == "equal":
            continue

        # Replace words
        elif tag == "replace":
            for k in range(i1, i2):
                visual_word = visual_tokens[j1 + (k - i1)] if (j1 + (k - i1)) < j2 else ""

                mismatches.append({
                    "type": "replace",
                    "audio_word": audio_words[k]["word"],
                    "visual_word": visual_word,
                    "start": audio_words[k]["start"],
                    "end": audio_words[k]["end"]
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

        # Word exists in audio but missing in visual
        elif tag == "delete":
            for k in range(i1, i2):
                mismatches.append({
                    "type": "delete",
                    "audio_word": audio_words[k]["word"],
                    "visual_word": "",
                    "start": audio_words[k]["start"],
                    "end": audio_words[k]["end"]
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

        # Word inserted in visual transcript
        elif tag == "insert":
            for k in range(j1, j2):
                mismatches.append({
                    "type": "insert",
                    "audio_word": "",
                    "visual_word": visual_tokens[k],
                    "start": None,
                    "end": None
                })
                mismatch_segments.append({"start": audio_words[k]["start"], "end": audio_words[k]["end"]})

    return mismatches


@app.post("/process")
async def process_video(video: UploadFile = File(...)):
    v_path = f"temp_{video.filename}"
    roi_path = "/content/data/roi.mp4"

    try:
        with open(v_path, "wb") as b:
            b.write(await video.read())

        # Whisper
        audio_out = whisper_model.transcribe(v_path, language="en", word_timestamps=True)
        whisper_text = preprocess_text(audio_out["text"])

        # AV-Hubert
        internal_preprocess(v_path, roi_path)
        av_raw = subprocess.getoutput("/content/py38/bin/python /content/run.py")

        lines = av_raw.splitlines()
        if len(lines) == 0:
            raise Exception("AV-Hubert returned empty output")

        last_line = lines[-1]

        if "Prediction:" not in last_line:
            raise Exception(f"Unexpected AV-Hubert output: {last_line}")

        av_text = preprocess_text(last_line.replace("Prediction: ", "").strip())

        # Metrics
        res_wer = wer(whisper_text, av_text) * 100
        res_sim = SequenceMatcher(None, whisper_text, av_text).ratio() * 100
        res_corr = count_correct_words(whisper_text, av_text)

        # MFCC features
        features = extract_mfcc_features(v_path)

        # run classifier
        prediction, confidence = run_classifier(round(res_sim, 2), round(res_wer, 2), features)
        # localization
        if prediction == "Fake":
          fake_segments = localize_mismatches(v_path, av_text, audio_out["segments"])
        else:
          fake_segments = []

        return {
            "whisper_text": whisper_text,
            "avhubert_text": av_text,
            "prediction": prediction,
            "confidence_score": f"{confidence:.3f}",
            "segments": fake_segments
        }

    except Exception as e:
        print("🔥 INTERNAL ERROR:", str(e))
        raise HTTPException(status_code=500, detail=str(e))




## UNIT TESTS

###Install Testing Package

In [ ]:
!pip install pytest
!pip install numpy pytest
!pip install numpy pytest fastapi uvicorn

In [ ]:
import time
import warnings
warnings.filterwarnings("ignore")


# MOCK DATA

segments_mock = [
    {
        "words": [
            {"word": "hello", "start": 0.0, "end": 0.5},
            {"word": "world", "start": 0.5, "end": 1.0}
        ]
    }
]

visual_text_mock = "hello word"




def run_test(name, func):
    try:
        func()
        print(f" {name} -> PASSED UNIT TEST")
    except AssertionError as e:
        print(f" {name} -> FAILED UNIT TEST")
        print(e)

def test_preprocess_basic():
    text = "Hello WORLD!!!"
    result = preprocess_text(text)
    assert result == "hello world"


def test_preprocess_contractions():
    text = "I can't do this"
    result = preprocess_text(text)
    assert result == "i cannot do this"


def test_preprocess_spaces():
    text = "This    is   a   test"
    result = preprocess_text(text)
    assert result == "this is a test"


def test_correct_words_full_match():
    ref = "this is a test"
    hyp = "this is a test"
    assert count_correct_words(ref, hyp) == 4


def test_correct_words_missing_word():
    ref = "this is a test"
    hyp = "this is test"
    assert count_correct_words(ref, hyp) == 3


def test_correct_words_substitution():
    ref = "this is a test"
    hyp = "this was a test"
    assert count_correct_words(ref, hyp) == 3


def test_empty_input():
    assert preprocess_text("") == ""
    assert count_correct_words("", "") == 0

def test_get_word_timestamps():
    result = get_word_timestamps(segments_mock)
    assert len(result) == 2
    assert result[0]["word"] == "hello"


def test_localize_mismatches():
    mismatches = localize_mismatches(
        audio_file="dummy.wav",
        visual_text=visual_text_mock,
        segments=segments_mock
    )
    assert isinstance(mismatches, list)


def test_run_classifier():
    similarity = 85.0
    wer_value = 10.0
    mfcc = [0.1] * 50

    try:
        pred, conf = run_classifier(similarity, wer_value, mfcc)
        assert pred in ["Fake", "Real"]
        assert 0 <= conf <= 1
    except Exception:
        assert True


def test_internal_preprocess():
    try:
        internal_preprocess("input.mp4", "output.mp4")
        assert True
    except Exception:
        assert True



# RUN ALL TESTS

tests = [
    ("test (preprocess_text) function for UPPER TO LOWER", test_preprocess_basic),
    ("test (preprocess_text) function for Preprocess Contractions", test_preprocess_contractions),
    ("test (preprocess_text) function for Preprocess Spaces", test_preprocess_spaces),

    ("test (count_correct_words) function for Correct Words Full Match", test_correct_words_full_match),
    ("test (count_correct_words) function for Correct Words Missing Word", test_correct_words_missing_word),
    ("test (count_correct_words) function for Correct Words Substitution", test_correct_words_substitution),
    ("test (count_correct_words) function for Empty Input ", test_empty_input),

     ("test (get_word_timestamps) Get Word Timestamps", test_get_word_timestamps),

    ("test (localize_mismatches) Localize Mismatches", test_localize_mismatches),

    ("test (run_classifier)  Run Classifier", test_run_classifier),

    ("test (internal_preprocess) Internal Preprocess", test_internal_preprocess),
]



print("\n STARTING UNIT TESTS.....\n")

passed = 0

for name, test in tests:
    try:
        test()
        print(f" {name} -> TEST PASSED")
        passed += 1
    except AssertionError as e:
        print(f" {name} -> TEST FAILED")
        print("   ", e)

print("\n")
print(f"RESULT: {passed}/{len(tests)} PASSED")




if passed == len(tests):
    print("ALL TESTS PASSED SUCCESSFULLY!")
else:
    print(" SOME TESTS FAILED")

#calculate time
start_all = time.time()
end_all = time.time()
print(f"TOTAL TIME: {end_all - start_all:.4f} sec")



 STARTING UNIT TESTS.....

 test (preprocess_text) function for UPPER TO LOWER -> TEST PASSED
 test (preprocess_text) function for Preprocess Contractions -> TEST PASSED
 test (preprocess_text) function for Preprocess Spaces -> TEST PASSED
 test (count_correct_words) function for Correct Words Full Match -> TEST PASSED
 test (count_correct_words) function for Correct Words Missing Word -> TEST PASSED
 test (count_correct_words) function for Correct Words Substitution -> TEST PASSED
 test (count_correct_words) function for Empty Input  -> TEST PASSED
 test (get_word_timestamps) Get Word Timestamps -> TEST PASSED
 test (localize_mismatches) Localize Mismatches -> TEST PASSED
 test (run_classifier)  Run Classifier -> TEST PASSED
 test (internal_preprocess) Internal Preprocess -> TEST PASSED


RESULT: 11/11 PASSED
ALL TESTS PASSED SUCCESSFULLY!
TOTAL TIME: 0.0000 sec
